In [1]:
"""
Benchmark: BB84 WCP Decoy — correct pipeline (Winnick et al.)
  Curve 1: 2-intensity analytic bound (Ma et al. 2005)
  Curve 2: 2-intensity LP  + FW2StepSolver (squashed qubit)
  Curve 3: 3-intensity LP  + FW2StepSolver (squashed qubit)

Sweep: eta = 10^linspace(0, -4, 20), fixed mu/nu1/nu2/d/e_d/pz.

Produces:
  benchmark_decoy_fw2.csv
  benchmark_decoy_fw2_curves.png       — key rate vs eta (log-log, inverted)
  benchmark_decoy_fw2_improvement.png  — gain of FW2 over analytic (semilogy)
"""

import sys, numpy as np, csv
sys.path.insert(0, r"C:\Users\Mario\Documents\OpenQKD\openqkd_python")

import matplotlib.pyplot as plt
from openqkd.presets.bb84_wcp_decoy_preset import BasicBB84WCPDecoyPreset
from openqkd.optimizer.main_iteration      import MainIteration


# ── Closed-form analytic baseline (Ma et al. 2005, 2-intensity) ───────────
def _gain(mu, eta, d):
    return 1.0 - (1.0 - d) * np.exp(-eta * mu)

def _qber(mu, eta, d, e_d):
    Q = _gain(mu, eta, d)
    if Q <= 0: return 0.5
    return (0.5 * d * np.exp(-mu) + e_d * (Q - d * np.exp(-mu))) / Q

def h(p):
    if p <= 0 or p >= 1: return 0.0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def keyrate_analytic_2int(eta, mu, nu, d, e_d, fEC, pz):
    Q_mu = _gain(mu,  eta, d);  E_mu = _qber(mu,  eta, d, e_d)
    Q_nu = _gain(nu,  eta, d);  E_nu = _qber(nu,  eta, d, e_d)
    Y0   = d
    denom = mu * nu - nu**2
    if denom <= 0: return 0.0
    Y1_L = (mu**2 * np.exp(-mu) / denom) * (
        Q_nu * np.exp(nu)
        - Q_mu * np.exp(mu) * (nu**2 / mu**2)
        - (1.0 - nu**2 / mu**2) * Y0
    )
    Y1_L = max(Y1_L, 0.0)
    if Y1_L <= 0: return 0.0
    e1_U = min(max(
        (E_nu * Q_nu * np.exp(nu) - 0.5 * Y0) / (Y1_L * nu), 0.0), 0.5)
    Q1_L = Y1_L * mu * np.exp(-mu)
    sift = pz**2 + (1 - pz)**2
    return max(sift * Q1_L * (1.0 - h(e1_U)) - Q_mu * fEC * h(E_mu), 0.0)


# ── Protocol parameters ────────────────────────────────────────────────────
MU   = 0.5
NU1  = 0.1
NU2  = 0.01
D    = 1e-6
E_D  = 0.03
PZ   = 0.5
FEC  = 1.16

ETA_RANGE = 10 ** np.linspace(0, -4, 20)

# ── Sweep ──────────────────────────────────────────────────────────────────
print(f"{'eta':>12} | {'2-int analytic':>14} | {'2-int FW2':>14} | {'3-int FW2':>14}")
print("-" * 64)

rows      = []
kr_anal   = []
kr_fw2_2  = []
kr_fw2_3  = []

for eta in ETA_RANGE:
    # Curve 1 — analytic
    r_an = keyrate_analytic_2int(eta, MU, NU1, D, E_D, FEC, PZ)

    # Curve 2 — 2-intensity LP + FW2StepSolver
    r2 = MainIteration(BasicBB84WCPDecoyPreset(
        eta=eta, intensities=[MU, NU1], d=D, e_d=E_D, pz=PZ, fEC=FEC
    ))["key_rate"]

    # Curve 3 — 3-intensity LP + FW2StepSolver
    r3 = MainIteration(BasicBB84WCPDecoyPreset(
        eta=eta, intensities=[MU, NU1, NU2], d=D, e_d=E_D, pz=PZ, fEC=FEC
    ))["key_rate"]

    kr_anal.append(r_an)
    kr_fw2_2.append(r2)
    kr_fw2_3.append(r3)

    rows.append({
        "eta":           round(float(eta), 8),
        "analytic_2int": round(r_an, 8),
        "fw2_2int":      round(r2,   8),
        "fw2_3int":      round(r3,   8),
        "gain_fw2_anal": round(r3 - r_an, 8),
    })
    print(f" {eta:12.6e} | {r_an:14.6e} | {r2:14.6e} | {r3:14.6e}")

# ── CSV ────────────────────────────────────────────────────────────────────
with open("benchmark_decoy_fw2.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader(); w.writerows(rows)

# ── Plot 1: key rate vs eta (log-log, inverted, no grid) ──────────────────
fig1, ax1 = plt.subplots(figsize=(8, 5))

ax1.loglog(ETA_RANGE, kr_anal,  "r--", lw=1.5,
           label=f"2-int analytic (Ma 2005)  $\\mu$={MU}, $\\nu_1$={NU1}")
ax1.loglog(ETA_RANGE, kr_fw2_2, "g-s", ms=5, lw=1.6,
           label=f"2-int LP + FW2  $\\mu$={MU}, $\\nu_1$={NU1}")
ax1.loglog(ETA_RANGE, kr_fw2_3, "b-o", ms=5, lw=1.8,
           label=f"3-int LP + FW2  $\\mu$={MU}, $\\nu_1$={NU1}, $\\nu_2$={NU2}")

ax1.set_xlim(1.2, 7e-5)
ax1.set_xlabel("Transmittance $\\eta$", fontsize=12)
ax1.set_ylabel("Key rate (bits/round)", fontsize=12)
ax1.set_title(
    "BB84 WCP Decoy — Key Rate vs Transmittance\n"
    f"Analytic vs LP + FW2StepSolver  |  $e_d$={E_D}, $d$={D}, $f_{{EC}}$={FEC}",
    fontsize=12
)
ax1.legend(fontsize=9)
plt.tight_layout()
plt.savefig("benchmark_decoy_fw2_curves.png", dpi=150)
plt.show()

# ── Plot 2: improvement of 3-int FW2 over analytic (semilogy, inverted) ───
gains = [r["gain_fw2_anal"] for r in rows]
valid = [(eta, g) for eta, g in zip(ETA_RANGE, gains) if g > 0]

if valid:
    eta_v, gain_v = zip(*valid)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    ax2.semilogy(eta_v, gain_v, "b-o", ms=6, lw=1.8,
                 label="3-int FW2 $-$ 2-int analytic")
    ax2.set_xscale("log")
    ax2.set_xlim(1.2, 7e-5)
    ax2.set_xlabel("Transmittance $\\eta$", fontsize=12)
    ax2.set_ylabel("$\\Delta$ key rate (bits/round)", fontsize=12)
    ax2.set_title(
        "BB84 WCP Decoy — Tightness Improvement\n"
        "3-int LP + FW2StepSolver over 2-int analytic bound (Ma 2005)",
        fontsize=12
    )
    ax2.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("benchmark_decoy_fw2_improvement.png", dpi=150)
    plt.show()

max_gain = max(gains)
print("\n-> benchmark_decoy_fw2.csv exported")
print("-> benchmark_decoy_fw2_curves.png exported")
print("-> benchmark_decoy_fw2_improvement.png exported")
print(f"-> Max gain (3-int FW2 over analytic): {max_gain:.2e}")


         eta | 2-int analytic |      2-int FW2 |      3-int FW2
----------------------------------------------------------------


TypeError: fw2step_solver() missing 4 required positional arguments: 'constraints_fn', 'key_dim', 'Gamma', and 'gamma_stats'